In [0]:
# ============================================
# Cell 1: Imports
# ============================================
from pyspark.sql.functions import (
    col, window, count, sum as _sum, to_timestamp
)

# ============================================
# Cell 2: Config
# ============================================
silver_table = "workspace.default.silver_user_activity"

gold_funnel_table = "workspace.default.gold_funnel_metrics_hourly"
gold_revenue_table = "workspace.default.gold_revenue_hourly"

funnel_checkpoint = "/Volumes/workspace/default/ecommerce_data/checkpoints/gold_funnel_hourly"
revenue_checkpoint = "/Volumes/workspace/default/ecommerce_data/checkpoints/gold_revenue_hourly"

# Watermark tolerance — how late an event can arrive before we give up
# waiting for it and finalize the window. 30 min is reasonable for
# e-commerce clickstream data; tighten later once you see real skew.
watermark_delay = "30 minutes"
window_duration = "1 hour"

# ============================================
# Cell 3: Read Silver as a stream, convert timestamp
# ============================================
# timestamp_ms is a LONG (epoch millis) — window()/watermark need an
# actual TimestampType column, so we convert it once up front.
silver_stream = (
    spark.readStream
    .format("delta")
    .table(silver_table)
    .withColumn("event_time", to_timestamp(col("timestamp_ms") / 1000))
)

# ============================================
# Cell 4: Funnel metrics — count of each event_type per hourly window
# ============================================
funnel_agg = (
    silver_stream
    .withWatermark("event_time", watermark_delay)
    .groupBy(
        window(col("event_time"), window_duration),
        col("event_type")
    )
    .agg(count("*").alias("event_count"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("event_type"),
        col("event_count")
    )
)

funnel_query = (
    funnel_agg.writeStream
    .format("delta")
    .outputMode("append")   # safe with watermark — emits once window closes
    .option("checkpointLocation", funnel_checkpoint)
    .trigger(availableNow=True)
    .toTable(gold_funnel_table)
)

funnel_query.awaitTermination()
print(f"Funnel metrics written. Rows: {spark.table(gold_funnel_table).count()}")

# ============================================
# Cell 5: Revenue metrics — sum of amount + payment count per hourly window
# ============================================
# Re-read Silver as a fresh stream for the second aggregation.
# Each writeStream needs its own source-read + checkpoint; sharing one
# DataFrame across two independent streaming sinks isn't reliable here.
silver_stream_2 = (
    spark.readStream
    .format("delta")
    .table(silver_table)
    .withColumn("event_time", to_timestamp(col("timestamp_ms") / 1000))
)

revenue_agg = (
    silver_stream_2
    .withWatermark("event_time", watermark_delay)
    .filter(col("event_type") == "payment")
    .groupBy(
        window(col("event_time"), window_duration)
    )
    .agg(
        _sum("amount").alias("total_revenue"),
        count("*").alias("payment_count")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("total_revenue"),
        col("payment_count")
    )
)

revenue_query = (
    revenue_agg.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", revenue_checkpoint)
    .trigger(availableNow=True)
    .toTable(gold_revenue_table)
)

revenue_query.awaitTermination()
print(f"Revenue metrics written. Rows: {spark.table(gold_revenue_table).count()}")

# ============================================
# Cell 6: Verify both tables
# ============================================
print("=== Funnel metrics (sample) ===")
spark.table(gold_funnel_table).orderBy("window_start").show(20, truncate=False)

print("=== Revenue metrics (sample) ===")
spark.table(gold_revenue_table).orderBy("window_start").show(20, truncate=False)